# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Structured Streaming (sockets)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [5]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Examples on Structured Streaming",
                   master_url="spark://spark-master:7077")

su.spark

# Create a data stream from a local socket

### Install netcat utility

In [6]:
!apt-get update
!apt-get install -y netcat

Hit:1 http://ports.ubuntu.com/ubuntu-ports jammy InRelease
Hit:2 http://ports.ubuntu.com/ubuntu-ports jammy-updates InRelease
Hit:3 http://ports.ubuntu.com/ubuntu-ports jammy-backports InRelease
Hit:4 http://ports.ubuntu.com/ubuntu-ports jammy-security InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
netcat is already the newest version (1.218-4ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 58 not upgraded.


### Connect Spark to the socket

In [7]:
import pyspark.sql.functions as F

# Create the remote connection
lines = su.spark.readStream \
            .format("socket") \
            .option("host", "localhost") \
            .option("port", 9999) \
            .load()

# Perform some transformations to the input data (word counter)
words = lines.select(F.explode(F.split(lines.value, " ")).alias("word"))
word_count = words.groupBy("word").count()

# Send transformed data to the Sink
query = word_count.writeStream \
            .outputMode("complete") \
            .format("console") \
            .start()
query.awaitTermination(300)


26/03/26 03:40:05 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.
26/03/26 03:40:05 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-2f950138-14cf-4710-b7f0-1c5384a544b1. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/03/26 03:40:05 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
+----+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|mundo|    1|
| hola|    1|
+-----+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|   de|    1|
|nuevo|    1|
|mundo|    1|
| hola|    2|
+-----+-----+



False

26/03/26 06:57:52 WARN MicroBatchExecutionContext: Query progress update takes longer than batch processing time. Progress update takes 902628 milliseconds. Batch processing takes 0 milliseconds
26/03/26 12:32:28 WARN MicroBatchExecutionContext: Query progress update takes longer than batch processing time. Progress update takes 900704 milliseconds. Batch processing takes 0 milliseconds
26/03/26 15:40:28 WARN MicroBatchExecutionContext: Query progress update takes longer than batch processing time. Progress update takes 619028 milliseconds. Batch processing takes 0 milliseconds


In [8]:
su.spark.stop()